In [ ]:
import torch
import torch.nn as nn
from torchvision import models

# ============================================================
# MODEL 1 — Custom CNN (built from scratch)
# ============================================================
class CustomCNN(nn.Module):
    def __init__(self, num_classes=6):
        super(CustomCNN, self).__init__()

        # Block 1 — 3 → 32 channels
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 224x224x32
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 112x112x32
            nn.Dropout2d(0.25)
        )

        # Block 2 — 32 → 64 channels
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1), # 112x112x64
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                          # 56x56x64
            nn.Dropout2d(0.25)
        )

        # Block 3 — 64 → 128 channels
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, kernel_size=3, padding=1), # 56x56x128
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                           # 28x28x128
            nn.Dropout2d(0.25)
        )

        # Block 4 — 128 → 256 channels
        self.block4 = nn.Sequential(
            nn.Conv2d(128, 256, kernel_size=3, padding=1), # 28x28x256
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),                            # 14x14x256
            nn.Dropout2d(0.25)
        )

        # Classifier
        self.classifier = nn.Sequential(
            nn.Flatten(),                        # 14*14*256 = 50176
            nn.Linear(50176, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.classifier(x)
        return x


# ============================================================
# MODEL 2 — ResNet50 (pretrained)
# ============================================================
def build_resnet50(num_classes=6):
    model = models.resnet50(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False          # freeze backbone
    model.fc = nn.Sequential(
        nn.Linear(model.fc.in_features, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


# ============================================================
# MODEL 3 — VGG16 (pretrained)
# ============================================================
def build_vgg16(num_classes=6):
    model = models.vgg16(pretrained=True)
    for param in model.parameters():
        param.requires_grad = False          # freeze backbone
    model.classifier[6] = nn.Sequential(
        nn.Linear(4096, 256),
        nn.ReLU(),
        nn.Dropout(0.4),
        nn.Linear(256, num_classes)
    )
    return model


print("All 3 models defined!")

In [ ]:
from sklearn.metrics import classification_report
import time

def train_model(model, model_name, epochs=5):
    model = model.to(device)

    # Custom CNN trains all params, pretrained models train only head
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    save_path = f'/content/drive/MyDrive/ContentRecognition/checkpoints/{model_name}.pth'

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,}")
    print(f"{'='*50}")

    for epoch in range(epochs):
        start = time.time()
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            loader = train_loader if phase == 'train' else val_loader

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_corrects.double() / len(loader.dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                torch.save(model.state_dict(), save_path)

        elapsed = time.time() - start
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {history['train_acc'][-1]:.4f} | "
              f"Val Acc: {history['val_acc'][-1]:.4f} | "
              f"Time: {elapsed:.1f}s")

    print(f"\n✅ Best Val Acc ({model_name}): {best_val_acc:.4f}")
    return model, history, best_val_acc, save_path


# ── Run all 3 ──
results = {}

# Custom CNN
custom_cnn = CustomCNN(num_classes=6)
custom_cnn, hist_custom, acc_custom, path_custom = train_model(custom_cnn, "CustomCNN")
results['Custom CNN'] = acc_custom.item()

# ResNet50
resnet = build_resnet50(num_classes=6)
resnet, hist_resnet, acc_resnet, path_resnet = train_model(resnet, "ResNet50")
results['ResNet50'] = acc_resnet.item()

# VGG16
vgg = build_vgg16(num_classes=6)
vgg, hist_vgg, acc_vgg, path_vgg = train_model(vgg, "VGG16")
results['VGG16'] = acc_vgg.item()

In [ ]:
from sklearn.metrics import classification_report
import time

def train_model(model, model_name, epochs=5):
    model = model.to(device)

    # Custom CNN trains all params, pretrained models train only head
    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=0.001
    )
    scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    criterion = nn.CrossEntropyLoss()

    best_val_acc = 0.0
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    save_path = f'/content/drive/MyDrive/ContentRecognition/checkpoints/{model_name}.pth'

    print(f"\n{'='*50}")
    print(f"Training: {model_name}")
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Trainable params: {trainable:,}")
    print(f"{'='*50}")

    for epoch in range(epochs):
        start = time.time()
        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            loader = train_loader if phase == 'train' else val_loader

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss    = criterion(outputs, labels)
                    _, preds = torch.max(outputs, 1)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss     += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            if phase == 'train':
                scheduler.step()

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc  = running_corrects.double() / len(loader.dataset)

            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_acc'].append(epoch_acc.item())

            if phase == 'val' and epoch_acc > best_val_acc:
                best_val_acc = epoch_acc
                torch.save(model.state_dict(), save_path)

        elapsed = time.time() - start
        print(f"Epoch {epoch+1}/{epochs} | "
              f"Train Acc: {history['train_acc'][-1]:.4f} | "
              f"Val Acc: {history['val_acc'][-1]:.4f} | "
              f"Time: {elapsed:.1f}s")

    print(f"\n✅ Best Val Acc ({model_name}): {best_val_acc:.4f}")
    return model, history, best_val_acc, save_path


# ── Run all 3 ──
results = {}

# Custom CNN
custom_cnn = CustomCNN(num_classes=6)
custom_cnn, hist_custom, acc_custom, path_custom = train_model(custom_cnn, "CustomCNN")
results['Custom CNN'] = acc_custom.item()

# ResNet50
resnet = build_resnet50(num_classes=6)
resnet, hist_resnet, acc_resnet, path_resnet = train_model(resnet, "ResNet50")
results['ResNet50'] = acc_resnet.item()

# VGG16
vgg = build_vgg16(num_classes=6)
vgg, hist_vgg, acc_vgg, path_vgg = train_model(vgg, "VGG16")
results['VGG16'] = acc_vgg.item()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

class_names = ['buildings', 'forest', 'glacier', 'mountain', 'sea', 'street']

def evaluate_model(model, save_path, model_name):
    model.load_state_dict(torch.load(save_path))
    model.eval().to(device)

    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.numpy())

    print(f"\n{'='*50}")
    print(f"Classification Report — {model_name}")
    print(f"{'='*50}")
    print(classification_report(all_labels, all_preds, target_names=class_names))
    return all_preds, all_labels


# Evaluate all
preds_custom, labels = evaluate_model(custom_cnn, path_custom, "Custom CNN")
preds_resnet, _      = evaluate_model(resnet,      path_resnet, "ResNet50")
preds_vgg,    _      = evaluate_model(vgg,          path_vgg,   "VGG16")


# ── Plot 1: Accuracy comparison bar chart ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

model_names = list(results.keys())
accuracies  = [v * 100 for v in results.values()]
colors      = ['#E74C3C', '#2B5BA8', '#27AE60']

bars = axes[0].bar(model_names, accuracies, color=colors, width=0.5)
axes[0].set_title('Model Accuracy Comparison', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Validation Accuracy (%)')
axes[0].set_ylim([0, 105])
axes[0].grid(axis='y', alpha=0.3)
for bar, acc in zip(bars, accuracies):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{acc:.1f}%', ha='center', va='bottom', fontweight='bold')

# ── Plot 2: Training curves all 3 ──
axes[1].plot(hist_custom['val_acc'], label='Custom CNN',  color='#E74C3C', marker='o')
axes[1].plot(hist_resnet['val_acc'], label='ResNet50',    color='#2B5BA8', marker='s')
axes[1].plot(hist_vgg['val_acc'],    label='VGG16',       color='#27AE60', marker='^')
axes[1].set_title('Validation Accuracy per Epoch', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ContentRecognition/model_comparison.png', dpi=150)
plt.show()


# ── Plot 3: Confusion matrices for all 3 ──
fig, axes = plt.subplots(1, 3, figsize=(20, 5))
for ax, preds, name in zip(axes,
                            [preds_custom, preds_resnet, preds_vgg],
                            ['Custom CNN', 'ResNet50', 'VGG16']):
    cm = confusion_matrix(labels, preds)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    ax.set_title(f'{name} — Confusion Matrix', fontweight='bold')
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    plt.setp(ax.get_xticklabels(), rotation=45)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/ContentRecognition/confusion_matrices.png', dpi=150)
plt.show()


# ── Final summary table ──
print("\n" + "="*55)
print(f"{'MODEL':<15} {'VAL ACCURACY':>15} {'TRAINABLE PARAMS':>20}")
print("="*55)
for name, acc in results.items():
    print(f"{name:<15} {acc*100:>14.2f}%")
print("="*55)